# Hindsight episodes, opportunity labels and rolling allocation

Read-only explorer of `rl-v6-independent-episode-windows-rolling15-v1`. All profitable episodes remain supervised, independent of portfolio selection. Labels use future outcomes; they are **not causal model inputs**. HOLD/EXIT rows refer to an explicitly hypothetical held position. Allocation shares are hindsight sizing targets, not executed trades or profitability results.

Only train/development dates are accessible. Aug26 stays sealed. This notebook reads existing certified banks and three sparse sidecars per day; it creates no new shards. Local ticker metrics are not portfolio token metrics. Existing stop/target/replay contracts are unchanged; this explorer focuses on the requested entry/HOLD/exit labels.


In [ ]:
import os
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
from pathlib import Path, PureWindowsPath
import json, hashlib, sys
import numpy as np
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from zoneinfo import ZoneInfo
from IPython.display import display, clear_output
import ipywidgets as widgets

# Laptop falls back only to the same workstation artifact authority, never
# to raw source files. Change these explicit paths for another audited build.
RUNTIME_ROOT = Path('D:/TradingML/runtimes')
SIDECAR_NAME = 'rl-v6-episode-windows-aa9021599'
if not (RUNTIME_ROOT / SIDECAR_NAME / '2026-08-05/complete.json').is_file():
    RUNTIME_ROOT = Path(r'\\DESKTOP-SAAI85T\Workstation-D\TradingML\runtimes')
LABEL_ROOT = RUNTIME_ROOT / SIDECAR_NAME
DATASET_PATH = RUNTIME_ROOT / 'rl-v6-training-ready-c824232b4/audit-cert-repair-df08998e3/complete.json'
assert DATASET_PATH.is_file() and LABEL_ROOT.is_dir(), 'Certified runtime artifacts unavailable'
repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'research/rl_trading/v6').is_dir())
if str(repo) not in sys.path: sys.path.insert(0, str(repo))
from research.rl_trading.v6.episode_windows import VERSION
from research.rl_trading.v6.features import SCALAR_NAMES
from research.rl_trading.v6.session_data import open_session

NY = ZoneInfo('America/New_York')
dataset = json.loads(DATASET_PATH.read_text())
ENTRIES = {d['day']:d for d in dataset['days'] if d['role'] in ('train','development')}
assert '2026-08-26' not in ENTRIES

def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''): h.update(block)
    return h.hexdigest()

def source_path(value):
    p=PureWindowsPath(value); base=PureWindowsPath('D:/TradingML/runtimes')
    return RUNTIME_ROOT.joinpath(*p.relative_to(base).parts)

def clock(us):
    return pd.to_datetime(np.asarray(us),unit='us',utc=True).tz_convert(NY)

def table(frame,limit=50):
    print(f'{frame.height:,} rows; showing first {min(limit,frame.height):,}')
    display(pd.DataFrame(frame.head(limit).to_dict(as_series=False)))

print('Runtime:', RUNTIME_ROOT, '\nLabels:', LABEL_ROOT)


## Day and provenance

Choose a completed day. Every selected table and its original episode/candidate source is hash-verified. The summary distinguishes positive-probability entry timestamps from the hard representative label (`p >= 0.5`). The latter is only a diagnostic projection of a soft distribution.


In [ ]:
CACHE = {}
SESSIONS = {}

def load_day(day):
    if day not in ENTRIES: raise ValueError('Only audited train/development dates')
    if day in CACHE: return CACHE[day]
    entry=ENTRIES[day]; directory=LABEL_ROOT/day
    proof=json.loads((directory/'complete.json').read_text())
    assert proof['version']==VERSION and proof['status']=='audited_independent_episode_windows'
    assert proof['day']==day and proof['role']==entry['role']
    assert proof['bank_certificate_sha256']==entry['bank_certificate_sha256']
    assert proof['sealed_test_accessed'] is False
    frames={}
    for name in ('flat','held','allocation'):
        path=directory/(name+'.parquet')
        assert sha(path)==proof['files'][name]['sha256'], name
        frames[name]=pl.read_parquet(path)
        assert frames[name].height==proof['files'][name]['rows']
    source=source_path(entry['bank_root'])
    for name in ('episodes','candidates'):
        path=source/(name+'.parquet')
        assert sha(path)==proof[name+'_sha256'], name
        frames[name]=pl.read_parquet(path)
    flat=frames['flat']; held=frames['held']; allocation=frames['allocation']
    totals=allocation.group_by('decision_us').agg(pl.col('allocation_weight').sum().alias('total'))
    assert totals.filter((pl.col('total')-1).abs()>1e-12).is_empty()
    assert allocation.select('decision_us','episode_uid').n_unique()==allocation.height
    assert flat.filter((pl.col('enter_probability')>0)&(pl.col('peak_headroom')<proof['config']['min_peak_headroom'])).is_empty()
    CACHE[day]=(proof,frames)
    return CACHE[day]

DAY='2026-08-05'
proof, frames=load_day(DAY)
flat,held,allocation=(frames[n] for n in ('flat','held','allocation'))
summary=flat.group_by('episode_uid','listing_id').agg(
    pl.len().alias('observed_flat_clocks'),
    (pl.col('enter_probability')>0).sum().alias('positive_entry_clocks'),
    (pl.col('enter_probability')>=.5).sum().alias('hard_entry_clocks'),
    pl.col('sample_weight').sum().alias('flat_weight_mass'))
print(json.dumps(proof,indent=2))
table(summary.sort('positive_entry_clocks',descending=True))
print(summary.select(pl.col('positive_entry_clocks').mean().alias('mean_positive_entries_per_episode'),
                     pl.col('hard_entry_clocks').mean().alias('mean_hard_entries_per_episode')))


## Interactive episode view

Blue entry quality is paired with WAIT; orange exit quality is paired with ticker HOLD. The lower panel shows that allocation varies separately from opportunity quality. Source episode bounds and candidates are shown beneath the chart.

Enable “MACD bank” to verify/open the certified candle bank and add the actual causal MACD channels. This reads and hashes the existing bank once per selected day; it can be slower over the network. It never creates dense windows or downloads data.


In [ ]:
day_choice=widgets.Dropdown(options=sorted(d for d in ENTRIES if (LABEL_ROOT/d/'complete.json').is_file()),value=DAY,description='Day')
episode_choice=widgets.Dropdown(description='Episode',layout=widgets.Layout(width='95%'))
read_macd=widgets.Checkbox(value=False,description='MACD bank (hash-verified)')
output=widgets.Output()

def render_episode(*_):
    with output:
        clear_output(wait=True)
        day=day_choice.value; uid=episode_choice.value
        if uid is None: return
        proof,frames=load_day(day)
        f=frames['flat'].filter(pl.col('episode_uid')==uid).sort('time_us')
        h=frames['held'].filter(pl.col('episode_uid')==uid).sort('time_us')
        a=frames['allocation'].filter(pl.col('episode_uid')==uid).sort('decision_us')
        identity=f['listing_id'][0]
        fig,axes=plt.subplots(4 if read_macd.value else 3,1,figsize=(14,10),sharex=True)
        axes[0].plot(clock(f['time_us']),f['close'],label='Observed close')
        peak=(f['close'].max())
        axes[0].scatter(clock(f.filter(pl.col('enter_probability')>0)['time_us']),
            f.filter(pl.col('enter_probability')>0)['close'],s=15,label='Positive entry label')
        axes[0].set_ylabel('Price'); axes[0].legend()
        axes[1].plot(clock(f['time_us']),f['enter_probability'],label='ENTER')
        axes[1].plot(clock(f['time_us']),1-f['enter_probability'],label='WAIT',alpha=.45)
        axes[1].plot(clock(h['time_us']),h['exit_probability'],label='EXIT')
        axes[1].plot(clock(h['time_us']),1-h['exit_probability'],label='Ticker HOLD',alpha=.45)
        axes[1].set_ylim(-.03,1.03);axes[1].set_ylabel('Soft target');axes[1].legend(ncol=4)
        # Bound the displayed allocation to this episode's observed label path.
        view=a.filter((pl.col('decision_us')>=f['time_us'].min())&(pl.col('decision_us')<=f['time_us'].max()))
        axes[2].plot(clock(view['decision_us']),view['allocation_weight'],label='Rolling share')
        axes[2].set_ylabel('Allocation weight');axes[2].legend()
        if read_macd.value:
            if day not in SESSIONS:
                e=ENTRIES[day]
                SESSIONS.clear()  # At most one full-bank pair remains open.
                SESSIONS[day]=open_session(source_path(e['bank_root']),runtime_root=RUNTIME_ROOT,
                    previous_root=source_path(e['previous_root']))
            values=SESSIONS[day].bank.listing(identity)
            mask=(values.close_us>=f['time_us'].min())&(values.close_us<=f['time_us'].max())
            valid=values.scalar[:,SCALAR_NAMES.index('indicator_available')]==1
            mask &= valid
            for channel in ('macd_line_rel','macd_signal_rel'):
                axes[3].plot(clock(values.close_us[mask]),values.scalar[mask,SCALAR_NAMES.index(channel)],label=channel)
            axes[3].legend();axes[3].set_ylabel('Causal MACD / close')
        axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S',tz=NY))
        axes[-1].set_xlabel('America/New_York');fig.suptitle(uid);fig.tight_layout();plt.show();plt.close(fig)
        table(frames['episodes'].filter((pl.col('listing_id')==identity)&
            ((pl.col('session_date')+':'+pl.col('listing_id')+':'+pl.col('episode_id').cast(pl.String))==uid)))
        table(frames['candidates'].filter(pl.col('episode_uid')==uid).sort('time_us'),limit=200)
        table(f.select('time_us','close','enter_probability','peak_headroom','allocation_weight'),limit=200)
        table(h,limit=200)

def choose_day(*_):
    _,frames=load_day(day_choice.value)
    order=frames['flat'].group_by('episode_uid').agg((pl.col('enter_probability')>0).sum().alias('count')).sort('count',descending=True)
    episode_choice.options=order['episode_uid'].to_list()
    episode_choice.value=episode_choice.options[0]
    render_episode()

day_choice.observe(choose_day,names='value')
episode_choice.observe(render_episode,names='value')
read_macd.observe(render_episode,names='value')
choose_day()
display(widgets.VBox([day_choice,episode_choice,read_macd,output]))


## Market-wide algorithm at a chosen clock

The table shows **all** selected episodes in `[t,t+15s)`, sorted by their maximum guarded candidate score. Repeated candidate times count once. The pie/bar chart is only a bounded display; no rows are omitted from the underlying table. This is the budget-share target, not a cash-constrained executed portfolio.

Set `MARKET_TIME_US` to a clock from the episode table, or leave it `None` to use its first positive-entry clock. Episodes excluded at this particular clock still retain their independent opportunity labels elsewhere. Rolling selection does not overwrite them with WAIT.


In [ ]:
MARKET_TIME_US=None
DISPLAY_LIMIT=30
_,frames=load_day(day_choice.value)
f=frames['flat'].filter(pl.col('episode_uid')==episode_choice.value).sort('time_us')
positive=f.filter(pl.col('enter_probability')>0)
if MARKET_TIME_US is None:
    MARKET_TIME_US=int(positive['time_us'][0] if positive.height else f['time_us'][0])
market=frames['allocation'].filter(pl.col('decision_us')==MARKET_TIME_US).sort('allocation_score',descending=True)
print('Clock:',clock([MARKET_TIME_US])[0], 'Window ends:',clock([MARKET_TIME_US+15_000_000])[0])
print('Episodes:',market.height,'Sum weights:',market['allocation_weight'].sum())
# Full result stays available as `market`; presentation alone is bounded.
table(market,limit=DISPLAY_LIMIT)
fig,ax=plt.subplots(figsize=(14,5))
shown=market.head(DISPLAY_LIMIT)
ax.bar(np.arange(shown.height),shown['allocation_weight'].to_numpy())
ax.set_xticks(np.arange(shown.height),shown['episode_uid'].to_list(),rotation=90,fontsize=7)
ax.set_ylabel('Normalized share');ax.set_title(f'Showing {shown.height} of {market.height} episodes; full table available in market')
fig.tight_layout();plt.show();plt.close(fig)

# Independent labels for window members vs episodes absent from this window.
coverage=frames['flat'].group_by('episode_uid').agg((pl.col('enter_probability')>0).sum().alias('positive_labels'))
coverage=coverage.join(market.select('episode_uid').with_columns(pl.lit(True).alias('in_this_window')),
    on='episode_uid',how='left').with_columns(pl.col('in_this_window').fill_null(False))
table(coverage.group_by('in_this_window').agg(pl.len().alias('episodes'),pl.col('positive_labels').sum().alias('positive_entry_labels')))


## Execution-adjusted score comparison (bps)

The original score is retained unchanged and displayed as `old_score * 10,000`. It includes the original 30-second time discount. `netbps` is **undiscounted**, modeled matched round-trip profit after observed spread, 100 ms participation capacity and broker fees, divided by a fixed $1,000 proposed budget. Unfilled capital contributes zero. Missing execution evidence stays unavailable, never zero cost. This is hindsight label evidence, not a causal input or realized P&L.

Set `COST_ROOT` to the output of `prepare_execution_features`. Existing candle banks and opportunity labels are unchanged. New allocation uses qualifying netbps scores in the same rolling 15-second algorithm. The score-specific day/episode selectors show prepared days only. The included canary is explicitly bounded; a portfolio allocation comparison is shown only for complete-day score cohorts.


In [ ]:
COST_ROOT = RUNTIME_ROOT / 'rl-v6-execution-costs-bps-v1'
if not COST_ROOT.exists():
    COST_ROOT = RUNTIME_ROOT / 'rl-v6-execution-costs-f6a8e4dbf-canary'
    print('Showing explicitly bounded historical canary:', COST_ROOT)

def compare_execution_scores(day, episode, cost_root=COST_ROOT):
    directory=Path(cost_root)/day
    if not (directory/'complete.json').is_file():
        print('Execution-cost sidecar not prepared:', directory)
        print('Run the SELECT-only prepare_execution_features CLI; no substitute score is invented here.')
        return
    proof=json.loads((directory/'complete.json').read_text())
    assert proof['version']=='rl-v6-bps-execution-features-v1'
    assert proof['sealed_test_accessed'] is False and proof['day']==day
    assert proof['bank_certificate_sha256']==ENTRIES[day]['bank_certificate_sha256']
    scores_path=directory/'scores.parquet'
    assert sha(scores_path)==proof['files']['scores']['sha256']
    scores=pl.read_parquet(scores_path)
    selected=scores.filter(pl.col('episode_uid')==episode).sort('time_us')
    print('Scope:',proof.get('preparation_scope'), '| missing cost rows:',proof['missing_cost_rows'],
          '| probe budget:',proof['budget'], '| participation:',proof['participation'])
    if selected.is_empty():
        print('Selected episode has no scored candidates in this preparation scope.'); return
    fig,axes=plt.subplots(2,1,figsize=(14,8),sharex=True)
    x=[clock(t) for t in selected['time_us']]
    axes[0].plot(x,selected['old_score_bps'].to_numpy(),label='Original score (bps; 30s discount)',color='tab:blue',marker='o')
    axes[0].plot(x,selected['netbps'].to_numpy(),label='Net execution score (bps; undiscounted)',color='tab:orange',marker='.')
    axes[0].axhline(100,color='gray',linestyle=':',label='100 bps allocation threshold')
    axes[0].set_ylabel('Score / proposed capital (bps)');axes[0].legend();axes[0].grid(alpha=.25)
    if proof.get('preparation_scope')=='complete_day':
        old_proof=json.loads((LABEL_ROOT/day/'complete.json').read_text())
        old_path=LABEL_ROOT/day/'allocation.parquet'
        assert sha(old_path)==old_proof['files']['allocation']['sha256']
        old=pl.read_parquet(old_path).filter(pl.col('episode_uid')==episode).sort('decision_us')
        allocation_path=directory/'allocation_netbps.parquet'
        assert sha(allocation_path)==proof['files']['allocation_netbps']['sha256']
        new=pl.read_parquet(allocation_path).filter(pl.col('episode_uid')==episode).sort('decision_us')
        for frame,label,color in ((old,'Original allocation share','tab:blue'),(new,'Netbps allocation share','tab:orange')):
            axes[1].plot([clock(t) for t in frame['decision_us']],frame['allocation_weight'].to_numpy(),label=label,color=color)
        axes[1].legend()
    else:
        axes[1].text(.05,.5,'Bounded score diagnostic: portfolio allocation comparison requires the full market cohort.',transform=axes[1].transAxes)
    axes[1].set_ylabel('Portfolio allocation fraction');axes[1].set_xlabel('America/New_York completed clock')
    axes[1].grid(alpha=.25)
    pad=max(pd.Timedelta(seconds=1),(max(x)-min(x))/20)
    axes[1].set_xlim(min(x)-pad,max(x)+pad)
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S',tz=NY))
    fig.suptitle(f"{day} | {selected['ticker'][0]} | episode {episode.rsplit(':',1)[-1]} | old and netbps scores")
    fig.tight_layout();plt.show()
    table(selected.select('time_us','old_score_bps','netbps','execution_cost_available','entry_capacity','exit_capacity'))

score_days=sorted(d.name for d in COST_ROOT.iterdir() if d.is_dir() and d.name in ENTRIES and (d/'complete.json').is_file()) if COST_ROOT.exists() else []
if score_days:
    score_day_choice=widgets.Dropdown(options=score_days,description='Score day')
    score_episode_choice=widgets.Dropdown(description='Episode',layout=widgets.Layout(width='90%'))
    score_output=widgets.Output()
    def choose_score_day(change=None):
        day=score_day_choice.value;proof=json.loads((COST_ROOT/day/'complete.json').read_text())
        path=COST_ROOT/day/'scores.parquet';assert sha(path)==proof['files']['scores']['sha256']
        rows=pl.read_parquet(path).sort('execution_cost_available',descending=True)
        options=rows['episode_uid'].unique(maintain_order=True).to_list()
        score_episode_choice.options=options
        if score_episode_choice.value not in options:score_episode_choice.value=options[0] if options else None
    def render_scores(change=None):
        score_output.clear_output(wait=True)
        with score_output:
            if score_episode_choice.value:compare_execution_scores(score_day_choice.value,score_episode_choice.value)
    score_day_choice.observe(choose_score_day,names='value')
    score_episode_choice.observe(render_scores,names='value')
    choose_score_day();display(score_day_choice,score_episode_choice,score_output);render_scores()
else:
    print('No prepared execution scores at',COST_ROOT)
